# Test – `extract_llm_answers.py`

Each section tests one extraction function using sample LLM outputs taken
directly from the prompt definitions in `prompt_list.py` and `prompt_list2.py`.

A **PASS / FAIL** line is printed after each assertion.

In [1]:
from extract_llm_answers import (
    extract_nlg_queries,
    extract_relations_with_scores,
    extract_sufficiency_decision,
    extract_subobjectives,
    extract_relation_chain,
    extract_entity_scores,
    extract_decision_json,
    extract_pruned_entities,
    extract_gog_answer,
)

def check(label, condition, result):
    status = "PASS" if condition else "FAIL"
    print(f"[{status}] {label}")
    if not condition:
        print(f"       got: {result!r}")

---
## 1 · `extract_nlg_queries` — `system_prompt` output

In [2]:
# ── happy path: clean list literal (matches the prompt example exactly) ──
raw_clean = '''["What are the major fields of study of students and graduates from Wesleyan University?",
"What did students who attended Wesleyan University major in?",
"What subjects did Wesleyan University students and graduates study as their major field?"]'''

r = extract_nlg_queries(raw_clean)
print("Result:", r)
check("returns 3 items", len(r) == 3, r)
check("first item correct", "Wesleyan" in r[0], r)

# ── LLM wraps output in a markdown code fence ──
raw_fenced = '''```python
["Query A?", "Query B?", "Query C?"]
```'''
r2 = extract_nlg_queries(raw_fenced)
print("Fenced result:", r2)
check("code fence stripped", r2 == ["Query A?", "Query B?", "Query C?"], r2)

# ── fallback: garbled but contains quoted strings ──
raw_garbled = 'Here are your queries: "First query?", \'Second query?\', "Third query?"'
r3 = extract_nlg_queries(raw_garbled)
print("Garbled result:", r3)
check("fallback finds 3 strings", len(r3) == 3, r3)

Result: ['What are the major fields of study of students and graduates from Wesleyan University?', 'What did students who attended Wesleyan University major in?', 'What subjects did Wesleyan University students and graduates study as their major field?']
[PASS] returns 3 items
[PASS] first item correct
Fenced result: ['Query A?', 'Query B?', 'Query C?']
[PASS] code fence stripped
Garbled result: ['First query?', 'Second query?', 'Third query?']
[PASS] fallback finds 3 strings


---
## 2 · `extract_relations_with_scores` — `extract_relation_prompt` output

In [3]:
# ── sample taken verbatim from the prompt's example answer ──
raw_relations = """1. {language/human_language/main_country (Score: 0.4)}: This relation is highly relevant as it directly relates to the country whose president is being asked for, and the main country where Brahui language is spoken in 1980.
2. {language/human_language/countries_spoken_in (Score: 0.3)}: This relation is also relevant as it provides information on the countries where Brahui language is spoken.
3. {base/rosetta/languoid/parent (Score: 0.2)}: This relation is less relevant but still provides some context on the language family.
<END>"""

r = extract_relations_with_scores(raw_relations)
print("Relations:", r)
check("3 relations parsed", len(r) == 3, r)
check("first relation name", r[0]["relation"] == "language/human_language/main_country", r)
check("first score is 0.4", r[0]["score"] == 0.4, r)
check("explanation non-empty", len(r[0]["explanation"]) > 0, r)
check("scores sum ~1.0", abs(sum(x["score"] for x in r) - 0.9) < 0.01, r)  # 0.4+0.3+0.2=0.9 per example

# ── no <END> marker ──
raw_no_end = "{some/relation (Score: 0.7)}: Very relevant.\n{other/relation (Score: 0.3)}: Less relevant."
r2 = extract_relations_with_scores(raw_no_end)
print("No-END result:", r2)
check("works without <END>", len(r2) == 2, r2)

Relations: [{'relation': 'language/human_language/main_country', 'score': 0.4, 'explanation': 'This relation is highly relevant as it directly relates to the country whose president is being asked for, and the main country where Brahui language is spoken in 1980. 2.'}, {'relation': 'language/human_language/countries_spoken_in', 'score': 0.3, 'explanation': 'This relation is also relevant as it provides information on the countries where Brahui language is spoken. 3.'}, {'relation': 'base/rosetta/languoid/parent', 'score': 0.2, 'explanation': 'This relation is less relevant but still provides some context on the language family.'}]
[PASS] 3 relations parsed
[PASS] first relation name
[PASS] first score is 0.4
[PASS] explanation non-empty
[PASS] scores sum ~1.0
No-END result: [{'relation': 'some/relation', 'score': 0.7, 'explanation': 'Very relevant.'}, {'relation': 'other/relation', 'score': 0.3, 'explanation': 'Less relevant.'}]
[PASS] works without <END>


---
## 3 · `extract_sufficiency_decision` — `prompt_evaluate` output

In [4]:
# ── {Yes} case with an embedded answer token ──
raw_yes = """{Yes}. Based on the given knowledge triplets, Rift Valley Province is located in Kenya, 
which uses the Kenyan shilling as its currency. Therefore, the answer to the question is {Kenyan shilling}.
<END>"""

r = extract_sufficiency_decision(raw_yes)
print("Yes case:", r)
check("is_sufficient True", r["is_sufficient"] is True, r)
check("answer extracted", r["answer"] == "Kenyan shilling", r)
check("<END> stripped from explanation", "<END>" not in r["explanation"], r)

# ── {No} case ──
raw_no = """{No}. Based on the given knowledge triplets, it's not sufficient to answer the entire question.
<END>"""
r2 = extract_sufficiency_decision(raw_no)
print("No case:", r2)
check("is_sufficient False", r2["is_sufficient"] is False, r2)
check("answer is None", r2["answer"] is None, r2)

# ── {Yes} without a second answer token ──
raw_yes_no_answer = "{Yes}. The triplets are sufficient."
r3 = extract_sufficiency_decision(raw_yes_no_answer)
print("Yes without answer token:", r3)
check("is_sufficient True, answer None", r3["is_sufficient"] and r3["answer"] is None, r3)

Yes case: {'is_sufficient': True, 'answer': 'Kenyan shilling', 'explanation': '{Yes}. Based on the given knowledge triplets, Rift Valley Province is located in Kenya, \nwhich uses the Kenyan shilling as its currency. Therefore, the answer to the question is {Kenyan shilling}.'}
[PASS] is_sufficient True
[PASS] answer extracted
[PASS] <END> stripped from explanation
No case: {'is_sufficient': False, 'answer': None, 'explanation': "{No}. Based on the given knowledge triplets, it's not sufficient to answer the entire question."}
[PASS] is_sufficient False
[PASS] answer is None
Yes without answer token: {'is_sufficient': True, 'answer': None, 'explanation': '{Yes}. The triplets are sufficient.'}
[PASS] is_sufficient True, answer None


---
## 4 · `extract_subobjectives` — `subobjective_prompt` output

In [5]:
# ── clean list literal (matches the prompt example) ──
raw_sub = "['Search the countries in the Caribbean', 'Search the country calling code for each Caribbean country', 'Compare the country calling codes to find the smallest one']"

r = extract_subobjectives(raw_sub)
print("Subobjectives:", r)
check("3 subobjectives", len(r) == 3, r)
check("first step correct", "Caribbean" in r[0], r)

# ── with code fence ──
raw_fenced = "```\n['Step A', 'Step B']\n```"
r2 = extract_subobjectives(raw_fenced)
print("Fenced:", r2)
check("fenced parsed", r2 == ["Step A", "Step B"], r2)

# ── double-quoted variant ──
raw_double = '["Find the author", "Find place of birth"]'
r3 = extract_subobjectives(raw_double)
print("Double-quoted:", r3)
check("double-quoted parsed", len(r3) == 2, r3)

Subobjectives: ['Search the countries in the Caribbean', 'Search the country calling code for each Caribbean country', 'Compare the country calling codes to find the smallest one']
[PASS] 3 subobjectives
[PASS] first step correct
Fenced: ['Step A', 'Step B']
[PASS] fenced parsed
Double-quoted: ['Find the author', 'Find place of birth']
[PASS] double-quoted parsed


---
## 5 · `extract_relation_chain` — `subobjective_prompt2` output

In [6]:
# ── matches the prompt example ──
raw_chain = "['located_in', 'has_calling_code']"

r = extract_relation_chain(raw_chain)
print("Chain:", r)
check("2 relations", len(r) == 2, r)
check("first relation", r[0] == "located_in", r)
check("second relation", r[1] == "has_calling_code", r)

# ── longer chain ──
raw_long = "['nationality', 'born_in', 'located_in_continent']"
r2 = extract_relation_chain(raw_long)
print("Long chain:", r2)
check("3-hop chain", len(r2) == 3, r2)

Chain: ['located_in', 'has_calling_code']
[PASS] 2 relations
[PASS] first relation
[PASS] second relation
Long chain: ['nationality', 'born_in', 'located_in_continent']
[PASS] 3-hop chain


---
## 6 · `extract_entity_scores` — `score_entity_candidates_prompt` output

In [7]:
# ── matches the prompt example ──
raw_scores = """Score: 0.0, 1.0, 0.0, 0.0, 0.0, 0.0
The movie that matches the given criteria is \"So Undercover\" with Miley Cyrus."""

r = extract_entity_scores(raw_scores, num_entities=6)
print("Scores:", r)
check("6 scores", len(r) == 6, r)
check("sum is 1.0", abs(sum(r) - 1.0) < 1e-9, r)
check("second entity gets score 1.0", r[1] == 1.0, r)

# ── comma-separated on its own line ──
raw_bare = "0.2, 0.5, 0.3\nSome explanation follows."
r2 = extract_entity_scores(raw_bare, num_entities=3)
print("Bare scores:", r2)
check("3 scores parsed", len(r2) == 3, r2)
check("values correct", r2 == [0.2, 0.5, 0.3], r2)

# ── uniform fallback when no parseable line found ──
r3 = extract_entity_scores("No numbers here at all, just text.", num_entities=4)
print("Fallback scores:", r3)
check("fallback uniform", r3 == [0.25, 0.25, 0.25, 0.25], r3)

Scores: [0.0, 1.0, 0.0, 0.0, 0.0, 0.0]
[PASS] 6 scores
[PASS] sum is 1.0
[PASS] second entity gets score 1.0
Bare scores: [0.2, 0.5, 0.3]
[PASS] 3 scores parsed
[PASS] values correct
Fallback scores: [0.25, 0.25, 0.25, 0.25]
[PASS] fallback uniform


---
## 7 · `extract_decision_json` — `decision_prompt` output

In [8]:
import json

# ── is_sufficient = false case ──
raw_not_suf = json.dumps({
    "reasoning": "The context shows Steve Bisciotti owns the Baltimore Ravens but does not mention the coach.",
    "is_sufficient": False,
    "action_plan": {
        "existing_relations": ["sports.professional_sports_team.owner_s"],
        "missing_relation_desc": "the head coach of the Baltimore Ravens"
    },
    "final_answer": None
})

r = extract_decision_json(raw_not_suf)
print("Decision (not sufficient):", r)
check("is_sufficient False", r["is_sufficient"] is False, r)
check("final_answer is None", r["final_answer"] is None, r)
check("action_plan present", "action_plan" in r, r)

# ── is_sufficient = true case ──
raw_suf = json.dumps({
    "reasoning": "The triplets directly show Kenya uses the Kenyan shilling.",
    "is_sufficient": True,
    "action_plan": {"existing_relations": [], "missing_relation_desc": None},
    "final_answer": "Kenyan shilling"
})

r2 = extract_decision_json(raw_suf)
print("Decision (sufficient):", r2)
check("is_sufficient True", r2["is_sufficient"] is True, r2)
check("final_answer correct", r2["final_answer"] == "Kenyan shilling", r2)

# ── wrapped in markdown code fence ──
raw_fenced = '```json\n' + raw_not_suf + '\n```'
r3 = extract_decision_json(raw_fenced)
print("Fenced decision:", r3)
check("fenced JSON parsed", "_parse_error" not in r3, r3)

# ── malformed JSON falls back gracefully ──
r4 = extract_decision_json("This is not JSON at all.")
print("Parse error result:", r4)
check("_parse_error key present", "_parse_error" in r4, r4)

Decision (not sufficient): {'reasoning': 'The context shows Steve Bisciotti owns the Baltimore Ravens but does not mention the coach.', 'is_sufficient': False, 'action_plan': {'existing_relations': ['sports.professional_sports_team.owner_s'], 'missing_relation_desc': 'the head coach of the Baltimore Ravens'}, 'final_answer': None}
[PASS] is_sufficient False
[PASS] final_answer is None
[PASS] action_plan present
Decision (sufficient): {'reasoning': 'The triplets directly show Kenya uses the Kenyan shilling.', 'is_sufficient': True, 'action_plan': {'existing_relations': [], 'missing_relation_desc': None}, 'final_answer': 'Kenyan shilling'}
[PASS] is_sufficient True
[PASS] final_answer correct
Fenced decision: {'reasoning': 'The context shows Steve Bisciotti owns the Baltimore Ravens but does not mention the coach.', 'is_sufficient': False, 'action_plan': {'existing_relations': ['sports.professional_sports_team.owner_s'], 'missing_relation_desc': 'the head coach of the Baltimore Ravens'},

---
## 8 · `extract_pruned_entities` — `entities_pruning_prompt` output

In [9]:
# ── clean JSON list ──
raw_ents = '["George Washington University", "Harvard University", "MIT"]'

r = extract_pruned_entities(raw_ents)
print("Entities:", r)
check("3 entities", len(r) == 3, r)
check("first entity", r[0] == "George Washington University", r)

# ── wrapped in code fence ──
raw_fenced = '```json\n["Thomas Jefferson", "John Adams"]\n```'
r2 = extract_pruned_entities(raw_fenced)
print("Fenced entities:", r2)
check("fenced parsed", r2 == ["Thomas Jefferson", "John Adams"], r2)

# ── list embedded in prose ──
raw_prose = 'The top entities are: ["Laura Ingalls Wilder", "De Smet"] based on the context.'
r3 = extract_pruned_entities(raw_prose)
print("Embedded list:", r3)
check("extracted from prose", len(r3) == 2, r3)
check("first entity correct", r3[0] == "Laura Ingalls Wilder", r3)

Entities: ['George Washington University', 'Harvard University', 'MIT']
[PASS] 3 entities
[PASS] first entity
Fenced entities: ['Thomas Jefferson', 'John Adams']
[PASS] fenced parsed
Embedded list: ['Laura Ingalls Wilder', 'De Smet']
[PASS] extracted from prose
[PASS] first entity correct


---
## 9 · `extract_gog_answer` — `GoG_answer_prompt` output

In [10]:
# ── {YES} case – taken verbatim from prompt example ──
raw_yes = '{YES}. Answer: ["George Washington University"]'

r = extract_gog_answer(raw_yes)
print("GoG YES:", r)
check("found True", r["found"] is True, r)
check("entity extracted", r["entities"] == ["George Washington University"], r)

# ── {NO} case ──
raw_no = '{NO}. Not specific information provided.'
r2 = extract_gog_answer(raw_no)
print("GoG NO:", r2)
check("found False", r2["found"] is False, r2)
check("entities empty", r2["entities"] == [], r2)

# ── multiple answer entities ──
raw_multi = '{YES}. Answer: ["Entity A", "Entity B", "Entity C"]'
r3 = extract_gog_answer(raw_multi)
print("GoG multi-answer:", r3)
check("3 entities", len(r3["entities"]) == 3, r3)
check("first entity", r3["entities"][0] == "Entity A", r3)

# ── case-insensitive YES ──
raw_lower = '{yes}. Answer: ["Some Entity"]'
r4 = extract_gog_answer(raw_lower)
print("GoG lowercase yes:", r4)
check("case-insensitive match", r4["found"] is True, r4)

GoG YES: {'found': True, 'entities': ['George Washington University'], 'raw': '{YES}. Answer: ["George Washington University"]'}
[PASS] found True
[PASS] entity extracted
GoG NO: {'found': False, 'entities': [], 'raw': '{NO}. Not specific information provided.'}
[PASS] found False
[PASS] entities empty
GoG multi-answer: {'found': True, 'entities': ['Entity A', 'Entity B', 'Entity C'], 'raw': '{YES}. Answer: ["Entity A", "Entity B", "Entity C"]'}
[PASS] 3 entities
[PASS] first entity
GoG lowercase yes: {'found': True, 'entities': ['Some Entity'], 'raw': '{yes}. Answer: ["Some Entity"]'}
[PASS] case-insensitive match


---
## Summary

In [11]:
print("All cells executed. Review PASS/FAIL lines above for results.")

All cells executed. Review PASS/FAIL lines above for results.
